In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x

In [2]:
# --- Cell 2: Offline install/import fix (NO internet) ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات التي نحتاجها (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet (عادة موجود في Kaggle)
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    # إذا مو موجود غالبًا Kaggle يوفره؛ لا نحاول تنزيله من النت
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).
✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
✅ Cell 2 ready.


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- Cell 22: V43 ULTIMATE (Lead Map + Dynamic Length + TTA + Per-Lead Sniper Gate + Robust ID Parse) ---
import os, gc, csv, glob, math
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, find_peaks, butter, filtfilt, resample_poly
import segmentation_models_pytorch as smp
from ultralytics import YOLO
from fractions import Fraction

# =========================
# 0) CONFIG
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/kaggle/input/physionet-ecg-image-digitization"
TEST_CSV_PATH = f"{DATA_DIR}/test.csv"
SAMPLE_PARQUET_PATH = f"{DATA_DIR}/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

YOLO_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
LEAD_TO_IDX = {n:i for i,n in enumerate(LEAD_NAMES)}
LEAD_SET = set(LEAD_NAMES)

TARGET_H = 256
MAX_W = 2048
YOLO_CONF = 0.12
MAX_CACHE = 16

# Sniper gate (تصفير على مستوى الليد فقط)
P2P_MIN = 0.03
P2P_MAX = 8.0
ROUGH_MAX = 0.45
CONF_MIN_YOLO = 0.20       # حد الثقة للليدات المكتشفة
CONF_MIN_GRID = 0.28       # حد أعلى للـ grid-fallback (لأنها قد تكون lead غلط)
USE_TTA = True
USE_DP_VITERBI = False     # True = أدق (أبطأ) | False = أسرع ومستقر

print(f"⚡ Device: {DEVICE}")

# =========================
# 1) Robust ID parsing (fix for 'Lengths mapped for 2 patients')
# =========================
def _norm_lead(s: str) -> str:
    s = str(s).strip()
    s = s.replace("Lead", "").replace("lead", "").replace(" ", "")
    s = s.replace("AVR","aVR").replace("AVL","aVL").replace("AVF","aVF")
    return s

def clean_pid(pid: str) -> str:
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def parse_row_id(rid: str):
    """
    Supports:
      - pid_idx_lead
      - pid_lead_idx
    where pid may contain underscores.
    """
    try:
        a,b,c = rid.rsplit("_", 2)
    except:
        return None

    b_lead = _norm_lead(b)
    c_lead = _norm_lead(c)

    # case 1: pid_idx_lead
    if b.isdigit() and c_lead in LEAD_SET:
        return clean_pid(a), int(b), c_lead

    # case 2: pid_lead_idx
    if c.isdigit() and b_lead in LEAD_SET:
        return clean_pid(a), int(c), b_lead

    # fallback: try numeric detection
    if b.isdigit():
        return clean_pid(a), int(b), c_lead if c_lead in LEAD_SET else None
    if c.isdigit():
        return clean_pid(a), int(c), b_lead if b_lead in LEAD_SET else None

    return None

# =========================
# 2) TEMPLATE & LENGTHS
# =========================
if not os.path.exists(SAMPLE_PARQUET_PATH):
    raise FileNotFoundError("❌ sample_submission.parquet not found")

print("📦 Reading template...")
template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = template["id"].astype(str).to_numpy()
del template
gc.collect()

print("🧠 Mapping patient lengths from template...")
patient_lengths = {}
bad_ids = 0
for rid in template_ids:
    parsed = parse_row_id(rid)
    if not parsed:
        bad_ids += 1
        continue
    pid, idx, lead = parsed
    if lead is None:
        bad_ids += 1
        continue
    cur = patient_lengths.get(pid, 0)
    if idx + 1 > cur:
        patient_lengths[pid] = idx + 1

print(f"✅ Lengths mapped for {len(patient_lengths):,} patients. (unparsed ids: {bad_ids})")

# =========================
# 3) INDEXING IMAGES + FS MAP
# =========================
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{DATA_DIR}/**/*.png", recursive=True) + glob.glob(f"{DATA_DIR}/**/*.jpg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map):,}")

def pick_id_fs_columns(df: pd.DataFrame):
    cols = list(df.columns)
    low = {c.lower(): c for c in cols}

    # pick id column
    id_candidates = ["record_id", "ecg_id", "id"]
    id_col = None
    for cand in id_candidates:
        if cand in low:
            id_col = low[cand]; break
    if id_col is None:
        # any column containing 'id'
        for k,v in low.items():
            if "id" in k:
                id_col = v; break

    # pick fs column
    fs_col = None
    for k,v in low.items():
        if k == "fs" or "sampling" in k or "sample" in k and "freq" in k or "hz" in k:
            fs_col = v; break
    if fs_col is None:
        for k,v in low.items():
            if "fs" in k:
                fs_col = v; break

    return id_col, fs_col

fs_map = {}
if os.path.exists(TEST_CSV_PATH):
    try:
        tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
        id_c, fs_c = pick_id_fs_columns(tdf)
        if id_c and fs_c:
            fs_map = dict(zip(tdf[id_c].astype(str).map(clean_pid), tdf[fs_c].astype(str)))
    except:
        pass
print(f"✅ fs_map loaded: {len(fs_map):,} items")

def get_image_path(pid: str):
    pid_s = clean_pid(pid)
    p = path_map.get(pid_s)
    if p:
        return p
    # sometimes filenames are numeric-only
    try:
        return path_map.get(str(int(float(pid_s))))
    except:
        return None

# =========================
# 4) MODELS + LEAD MAPPING
# =========================
print("⚙️ Loading models...")
yolo_model = None
CLASS_TO_LEAD_IDX = {i:i for i in range(12)}  # safe fallback identity
if os.path.exists(YOLO_PATH):
    try:
        yolo_model = YOLO(YOLO_PATH)
        if hasattr(yolo_model, "names") and isinstance(yolo_model.names, dict):
            tmp = {}
            for cid, cname in yolo_model.names.items():
                n = _norm_lead(cname)
                if n in LEAD_TO_IDX:
                    tmp[int(cid)] = LEAD_TO_IDX[n]
            # fill missing
            for i in range(12):
                tmp.setdefault(i, i)
            CLASS_TO_LEAD_IDX = tmp
        print(f"✅ YOLO loaded. CLASS_TO_LEAD_IDX: {CLASS_TO_LEAD_IDX}")
    except Exception as e:
        print("⚠️ YOLO load failed, using grid fallback only.")

unet_model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    decoder_attention_type="scse",
)
if os.path.exists(UNET_PATH):
    try:
        unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
    except:
        print("⚠️ UNet state_dict load failed (will run with random weights -> bad).")
unet_model.to(DEVICE).eval()
print("✅ UNet ready.")

# =========================
# 5) HELPERS
# =========================
def preprocess_remove_grid_rgb(img):
    # light grid removal (safe)
    try:
        out = img.copy()
        hsv = cv2.cvtColor(out, cv2.COLOR_RGB2HSV)
        mask = (cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
                cv2.inRange(hsv, (170, 50, 50), (180, 255, 255)))
        out[mask > 0] = 255
        return out
    except:
        return img

def robust_grid_spacing_px(img):
    """
    Estimate grid spacing in pixels (could be 1mm or 5mm).
    Works on RGB crop or full image.
    """
    try:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        # emphasize vertical grid lines
        edges = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sx = np.sum(np.abs(edges), axis=0)
        prom = np.percentile(sx, 85)
        peaks, _ = find_peaks(sx, distance=5, prominence=prom)
        if len(peaks) > 8:
            diffs = np.diff(peaks)
            diffs = diffs[(diffs > 3) & (diffs < 200)]
            if len(diffs) > 6:
                return float(np.median(diffs))
    except:
        pass
    return None

def pixels_per_mV_from_spacing(spacing_px):
    """
    Decide whether spacing is 1mm or 5mm by magnitude.
    - if spacing small (<18px): treat as 1mm (0.1mV) => pixels/mV = spacing*10
    - else: treat as 5mm (0.5mV) => pixels/mV = spacing*2
    """
    if spacing_px is None:
        return None
    if spacing_px < 18:
        return spacing_px * 10.0
    return spacing_px * 2.0

def apply_highpass(sig, fs):
    try:
        fs = float(fs)
        if fs <= 1:
            return sig
        nyq = 0.5 * fs
        if nyq <= 0:
            return sig
        b, a = butter(3, 0.5/nyq, btype="high")
        return filtfilt(b, a, sig)
    except:
        return sig

def resample_lp(sig, target_len):
    """Linear-phase resampling (host advice) via resample_poly."""
    n = len(sig)
    if n == 0:
        return sig
    if n == target_len:
        return sig.astype(np.float32)

    # rational approximation target_len/n
    frac = Fraction(target_len, n).limit_denominator(1000)
    up, down = frac.numerator, frac.denominator
    y = resample_poly(sig.astype(np.float32), up, down)
    # crop/pad exact length
    if len(y) > target_len:
        y = y[:target_len]
    elif len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y.astype(np.float32)

def viterbi_dp(prob_map, smooth=0.35):
    """DP Viterbi (optional) for a connected path."""
    H, W = prob_map.shape
    cost = -np.log(prob_map + 1e-6).astype(np.float32)
    dp = np.zeros((H, W), dtype=np.float32)
    parent = np.zeros((H, W), dtype=np.int16)
    dp[:, 0] = cost[:, 0]

    for t in range(1, W):
        prev = dp[:, t-1]
        up   = np.roll(prev, 1);   up[0]   = 1e9
        mid  = prev
        down = np.roll(prev, -1);  down[-1]= 1e9

        stack = np.stack([up + smooth, mid, down + smooth], axis=0)
        which = np.argmin(stack, axis=0)  # 0,1,2
        dp[:, t] = cost[:, t] + stack[which, np.arange(H)]
        parent[:, t] = np.clip(np.arange(H) + (which - 1), 0, H-1)

    path = np.zeros(W, dtype=np.int32)
    path[-1] = int(np.argmin(dp[:, -1]))
    for t in range(W-2, -1, -1):
        path[t] = parent[path[t+1], t+1]
    # invert y axis (top->bottom)
    return (H - path).astype(np.float32), path.astype(np.int32)

def extract_path(prob_map):
    """
    Returns:
      raw_y_inv (float): y path in "image coords" (larger=down) using inverted axis
      conf (float): mean probability along the chosen path
    """
    H, W = prob_map.shape
    if USE_DP_VITERBI:
        y_inv, y = viterbi_dp(prob_map)
        y = np.clip(y, 0, H-1)
        conf = float(np.mean(prob_map[y, np.arange(W)]))
        return y_inv, conf
    else:
        y = np.argmax(prob_map, axis=0).astype(np.int32)
        y_s = savgol_filter(y.astype(np.float32), 7, 2)
        y_s = np.clip(np.round(y_s).astype(np.int32), 0, H-1)
        conf = float(np.mean(prob_map[y_s, np.arange(W)]))
        y_inv = (H - y_s).astype(np.float32)
        return y_inv, conf

def lead_quality_ok(sig_mv, conf, is_yolo_crop):
    """Per-lead Sniper Gate."""
    if sig_mv is None or len(sig_mv) < 50:
        return False
    if not np.all(np.isfinite(sig_mv)):
        return False

    p95 = np.percentile(sig_mv, 95)
    p05 = np.percentile(sig_mv, 5)
    p2p = float(p95 - p05)
    if p2p < P2P_MIN or p2p > P2P_MAX:
        return False

    d2 = np.diff(sig_mv, 2)
    rough = float(np.mean(np.abs(d2))) / (p2p + 1e-6)
    if rough > ROUGH_MAX:
        return False

    conf_min = CONF_MIN_YOLO if is_yolo_crop else CONF_MIN_GRID
    if conf < conf_min:
        return False

    return True

def get_crops_with_fallback(img, model):
    """
    Returns:
      crops[12] RGB
      detected[12] bool (True if from YOLO)
    """
    h, w = img.shape[:2]
    crops = [None]*12
    detected = [False]*12

    # grid fallback
    rh, cw = h//4, w//3
    grid_crops = [img[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]
    for i in range(12):
        crops[i] = grid_crops[i]

    if model is None:
        return crops, detected

    try:
        scale = 1280 / max(h, w)
        if scale < 1.0:
            img_inf = cv2.resize(img, (int(w*scale), int(h*scale)))
        else:
            img_inf = img
            scale = 1.0

        res = model.predict(img_inf, conf=YOLO_CONF, verbose=False, device=DEVICE)
        if res and res[0].boxes:
            boxes = res[0].boxes.data.detach().cpu().numpy()
            for box in boxes:
                cls = int(box[5])
                tgt = CLASS_TO_LEAD_IDX.get(cls, cls)
                if 0 <= tgt < 12:
                    x1,y1,x2,y2 = box[:4] / scale
                    x1 = max(0, int(x1)); y1 = max(0, int(y1))
                    x2 = min(w, int(x2)); y2 = min(h, int(y2))
                    if x2 > x1+10 and y2 > y1+10:
                        crops[tgt] = img[y1:y2, x1:x2]
                        detected[tgt] = True
    except:
        pass

    return crops, detected

# =========================
# 6) COMPUTE PATIENT
# =========================
patient_cache = OrderedDict()

def compute_patient(pid, target_len):
    out = np.zeros((12, target_len), dtype=np.float32)

    path = get_image_path(pid)
    if not path:
        return out

    img = cv2.imread(path)
    if img is None:
        return out
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fs = float(fs_map.get(clean_pid(pid), 500.0))

    crops, detected = get_crops_with_fallback(img, yolo_model)

    # global spacing (optional)
    global_spacing = robust_grid_spacing_px(img)
    global_ppmv_orig = pixels_per_mV_from_spacing(global_spacing)

    # prepare batch for UNet (always 12)
    inputs = []
    scales = []
    clean_crops = []
    for i in range(12):
        c = crops[i]
        c_clean = preprocess_remove_grid_rgb(c)
        clean_crops.append(c_clean)

        h, w = c_clean.shape[:2]
        s = TARGET_H / max(h, 1)
        nw = int(w * s)
        if nw < 32: nw = 32
        if nw > MAX_W: nw = MAX_W

        rz = cv2.resize(c_clean, (nw, TARGET_H))
        t = torch.from_numpy(rz).permute(2,0,1).float() / 255.0
        inputs.append(t)
        scales.append(s)

    mw = max([t.shape[2] for t in inputs])
    mw = int(np.ceil(mw/32)*32)

    batch = torch.zeros((12, 3, TARGET_H, mw), device=DEVICE)
    for i, t in enumerate(inputs):
        batch[i, :, :, :t.shape[2]] = t.to(DEVICE)

    with torch.no_grad():
        if USE_TTA:
            p1 = torch.sigmoid(unet_model(batch))
            p2 = torch.sigmoid(unet_model(torch.flip(batch, [3])))
            preds = (p1 + torch.flip(p2, [3])) * 0.5
        else:
            preds = torch.sigmoid(unet_model(batch))
        preds = preds.detach().cpu().numpy()

    # per-lead extraction + per-lead sniper gate
    for i in range(12):
        w_orig = inputs[i].shape[2]
        prob = preds[i, 0, :, :w_orig]

        # path & confidence
        raw_y, conf = extract_path(prob)  # raw_y is inverted axis in pixel units

        # baseline center (IMPORTANT FIX: no "128")
        raw_centered = raw_y - np.median(raw_y)

        # grid spacing on crop (prefer local)
        local_spacing = robust_grid_spacing_px(clean_crops[i])
        ppmv_orig = pixels_per_mV_from_spacing(local_spacing)
        if ppmv_orig is None:
            ppmv_orig = global_ppmv_orig

        # convert to pixels-per-mV in resized space
        s = scales[i]
        if ppmv_orig is None:
            # fallback assume big-box ~25px => pixels/mV = 50px (orig) then scale
            ppmv_resized = 50.0 * s
        else:
            ppmv_resized = float(ppmv_orig) * s

        ppmv_resized = float(np.clip(ppmv_resized, 20.0, 2000.0))
        sig_mv = raw_centered / ppmv_resized

        # filters
        sig_mv = sig_mv - np.median(sig_mv)
        sig_mv = apply_highpass(sig_mv, fs)

        # gate
        if not lead_quality_ok(sig_mv, conf, detected[i]):
            continue  # keep zeros

        # resample (linear-phase)
        out[i] = resample_lp(sig_mv, target_len)

    return out

# =========================
# 7) WRITE SUBMISSION
# =========================
print("🚀 Writing submission.csv (V43 ULTIMATE)...")
with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for rid in tqdm(template_ids, desc="Processing"):
        parsed = parse_row_id(rid)
        if not parsed:
            writer.writerow([rid, "0.0000"])
            continue
        pid, idx, lead = parsed
        if lead is None:
            writer.writerow([rid, "0.0000"])
            continue

        t_len = patient_lengths.get(pid, 5000)

        if pid in patient_cache:
            mat = patient_cache[pid]
            patient_cache.move_to_end(pid)
        else:
            mat = compute_patient(pid, t_len)
            patient_cache[pid] = mat
            if len(patient_cache) > MAX_CACHE:
                patient_cache.popitem(last=False)

        lidx = LEAD_TO_IDX.get(lead, 0)
        val = 0.0
        if 0 <= idx < t_len:
            v = float(mat[lidx][idx])
            if np.isfinite(v):
                val = v

        writer.writerow([rid, f"{val:.4f}"])

del patient_cache
gc.collect()
torch.cuda.empty_cache()
print("✅ Done. submission.csv ready.")


⚡ Device: cuda
📦 Reading template...
🧠 Mapping patient lengths from template...
✅ Lengths mapped for 2 patients. (unparsed ids: 0)
🗂️ Indexing images...
✅ Indexed images: 8,795
✅ fs_map loaded: 2 items
⚙️ Loading models...
✅ YOLO loaded. CLASS_TO_LEAD_IDX: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}
✅ UNet ready.
🚀 Writing submission.csv (V43 ULTIMATE)...


Processing: 100%|██████████| 75000/75000 [00:03<00:00, 20428.99it/s]


✅ Done. submission.csv ready.


In [5]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.94 MB
🎉 Ready to Submit.
